# Redes Neuronales Recurrentes: cuando el orden importa

**Objetivo**: entender como procesa una RNN secuencias paso a paso,
por que el gradiente desaparece, como LSTM lo resuelve, y cuando NO usar
arquitecturas recurrentes.

In [1]:
import os, pathlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
IMG = pathlib.Path('images')
IMG.mkdir(exist_ok=True)
print('Setup OK')

Setup OK


## 1. El problema que resuelven las RNNs

Las redes densas (MLP) tratan cada ejemplo como independiente.
Pero hay datos donde el pasado importa: un ticket urgente despues de 3 tickets
similares tiene distinto significado que el mismo ticket aislado.
Las RNNs mantienen un **estado oculto** h_t que se actualiza en cada paso de la secuencia.

In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')

cell_color  = '#BBDEFB'
input_color = '#FFB74D'
state_color = '#A5D6A7'
text_color  = '#1A237E'

steps = 4
xs = [1.5, 4.5, 7.5, 10.5]
cell_w, cell_h = 2.0, 1.6
cell_y = 2.8

for i, x in enumerate(xs):
    box = FancyBboxPatch((x - cell_w/2, cell_y - cell_h/2),
                         cell_w, cell_h,
                         boxstyle='round,pad=0.12',
                         facecolor=cell_color, edgecolor='#1565C0', linewidth=1.8)
    ax.add_patch(box)
    ax.text(x, cell_y, f'RNN\nt={i}',
            ha='center', va='center', fontsize=11, color=text_color, fontweight='bold')

    # Input arrow from below
    ax.annotate('', xy=(x, cell_y - cell_h/2),
                xytext=(x, cell_y - cell_h/2 - 0.85),
                arrowprops=dict(arrowstyle='->', color=input_color, lw=2.0))
    ax.text(x, cell_y - cell_h/2 - 1.05, f'x_{i}',
            ha='center', va='top', fontsize=12, color=input_color, fontweight='bold')

    # Output arrow upward
    ax.annotate('', xy=(x, cell_y + cell_h/2 + 0.75),
                xytext=(x, cell_y + cell_h/2),
                arrowprops=dict(arrowstyle='->', color=state_color, lw=2.0))
    ax.text(x, cell_y + cell_h/2 + 0.88, f'h_{i}',
            ha='center', va='bottom', fontsize=12, color=state_color, fontweight='bold')

    # Horizontal arrow to next cell
    if i < steps - 1:
        ax.annotate('', xy=(xs[i+1] - cell_w/2, cell_y),
                    xytext=(x + cell_w/2, cell_y),
                    arrowprops=dict(arrowstyle='->', color='#546E7A', lw=2.0))

# h_{-1} initial state arrow
ax.annotate('', xy=(xs[0] - cell_w/2, cell_y),
            xytext=(xs[0] - cell_w/2 - 0.75, cell_y),
            arrowprops=dict(arrowstyle='->', color='#546E7A', lw=2.0))
ax.text(xs[0] - cell_w/2 - 0.85, cell_y, 'h_{-1}\n(ceros)',
        ha='right', va='center', fontsize=9, color='#546E7A')

# Formula
ax.text(7.0, 0.5,
        'h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)',
        ha='center', va='center', fontsize=11,
        color='#333', style='italic',
        bbox=dict(facecolor='#F5F5F5', edgecolor='#BDBDBD', boxstyle='round,pad=0.4'))

# Legend
patches = [
    mpatches.Patch(facecolor=cell_color,  edgecolor='#1565C0', label='Celula RNN'),
    mpatches.Patch(facecolor=input_color, label='Input x_t'),
    mpatches.Patch(facecolor=state_color, label='Estado oculto h_t'),
]
ax.legend(handles=patches, loc='upper right', fontsize=9, framealpha=0.9)

ax.set_title('Celula RNN desenrollada en el tiempo', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('images/B05B_fig01.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05B_fig01.png')

Guardado: images/B05B_fig01.png


> Antes de seguir: si la RNN procesa un mensaje de soporte técnico palabra a palabra,
> que información deberia 'recordar' la celula al final para clasificar si es urgente?

<details>
<summary>Orientacion para el instructor</summary>

Una respuesta madura menciona: palabras clave de urgencia ('caido', 'produccion', 'bloqueado'),
el tono acumulado, y que la RNN basica olvida el principio del mensaje si es largo.
Senal de comprension: el alumno identifica que la longitud del mensaje importa para
la capacidad de memoria de la celula.

</details>

## 2. El problema del gradiente que desaparece

En una RNN profunda (secuencias largas), el gradiente se multiplica por W_h en cada
paso hacia atras. Si |W_h| < 1, el gradiente se hace exponencialmente pequeño.
Si |W_h| > 1, explota. En la práctica, las RNNs simples no pueden aprender
dependencias mas alla de ~10-20 pasos.

In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

T = 20
pasos = np.arange(1, T + 1)
w_vals = [0.5, 0.9, 1.0, 1.1, 2.0]
colores = ['#1565C0', '#2E7D32', '#F9A825', '#E65100', '#880E4F']

fig, ax = plt.subplots(figsize=(10, 5))

for w, color in zip(w_vals, colores):
    grad = np.array([abs(w)**t for t in pasos])
    ax.semilogy(pasos, grad, color=color, linewidth=2.2,
                label=f'|W| = {w}')

ax.axhline(0.01, color='#777', linestyle='--', linewidth=1.4,
           label='Umbral: gradiente inutil (0.01)')
ax.set_xlabel('Pasos hacia atras (profundidad de la secuencia)', fontsize=11)
ax.set_ylabel('Magnitud del gradiente (escala log)', fontsize=11)
ax.set_title('Vanishing/exploding gradient en RNN segun |W|', fontsize=12, pad=10)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3, which='both')
ax.set_xlim(1, T)
plt.tight_layout()
plt.savefig('images/B05B_fig02.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05B_fig02.png')

Guardado: images/B05B_fig02.png


## 3. LSTM: memoria selectiva

LSTM (Long Short-Term Memory) añade tres compuertas a la celula RNN:

- **Forget gate**: que olvidar del estado anterior (sigmoide -> 0 = olvidar, 1 = recordar)
- **Input gate**: que informacion nueva añadir al estado
- **Output gate**: que parte del estado usar como salida

La clave: las compuertas son funciones aprendidas.
La red aprende que recordar para cada tarea.

In [4]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

def draw_box(ax, cx, cy, w, h, fc, ec, lbl, fs=9):
    box = FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                         boxstyle='round,pad=0.08',
                         facecolor=fc, edgecolor=ec, linewidth=1.6)
    ax.add_patch(box)
    ax.text(cx, cy, lbl, ha='center', va='center',
            fontsize=fs, color='#1A1A2E', fontweight='bold')

# Cell state line (top)
ax.annotate('', xy=(13.2, 5.5), xytext=(0.8, 5.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2.5))
ax.text(7.0, 5.85, 'Cell state  C_t  (memoria a largo plazo)',
        ha='center', va='center', fontsize=10, color='#2E7D32', fontweight='bold')

# Forget gate
draw_box(ax, 2.8, 4.0, 1.8, 1.0, '#EF9A9A', '#C62828', 'Forget\ngate (sigma)', fs=8)
ax.annotate('', xy=(2.8, 5.15), xytext=(2.8, 4.52),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.8))
ax.text(2.8, 5.3, 'x', ha='center', va='center', fontsize=16, color='#C62828', fontweight='bold')
ax.text(2.8, 2.8, 'f_t = sigma(W_f.[h,x]+b_f)',
        ha='center', va='center', fontsize=7.5, color='#B71C1C',
        bbox=dict(facecolor='#FFEBEE', edgecolor='#EF9A9A', boxstyle='round,pad=0.2'))
ax.annotate('', xy=(2.8, 3.5), xytext=(2.8, 2.98),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.4))

# Input gate
draw_box(ax, 6.2, 4.0, 1.8, 1.0, '#FFF59D', '#F57F17', 'Input\ngate (sigma)', fs=8)
draw_box(ax, 8.4, 4.0, 1.8, 1.0, '#FFF59D', '#F57F17', 'Candidate\n(tanh)', fs=8)
ax.annotate('', xy=(6.2, 5.15), xytext=(6.2, 4.52),
            arrowprops=dict(arrowstyle='->', color='#F57F17', lw=1.8))
ax.text(6.2, 5.3, '+', ha='center', va='center', fontsize=18, color='#F57F17', fontweight='bold')
ax.text(7.3, 2.8, 'i_t * c_hat:\nque anadir',
        ha='center', va='center', fontsize=7.5, color='#E65100',
        bbox=dict(facecolor='#FFF8E1', edgecolor='#FFF59D', boxstyle='round,pad=0.2'))

# Output gate
draw_box(ax, 11.4, 4.0, 1.8, 1.0, '#B2EBF2', '#006064', 'Output\ngate (sigma)', fs=8)
ax.annotate('', xy=(11.4, 5.15), xytext=(11.4, 4.52),
            arrowprops=dict(arrowstyle='->', color='#006064', lw=1.8))
ax.text(11.4, 5.3, 'x', ha='center', va='center', fontsize=16, color='#006064', fontweight='bold')
ax.text(11.4, 2.8, 'o_t: que mostrar\ncomo h_t',
        ha='center', va='center', fontsize=7.5, color='#004D40',
        bbox=dict(facecolor='#E0F7FA', edgecolor='#B2EBF2', boxstyle='round,pad=0.2'))

# h_t output arrow
ax.annotate('', xy=(13.2, 1.8), xytext=(11.4, 3.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.8))
ax.text(13.2, 1.6, 'h_t', ha='center', va='top', fontsize=12,
        color='#2E7D32', fontweight='bold')

# Input label
ax.text(0.5, 4.0, '[h_{t-1},\nx_t]', ha='center', va='center',
        fontsize=9, color='#37474F', style='italic')
ax.annotate('', xy=(1.85, 4.0), xytext=(1.3, 4.0),
            arrowprops=dict(arrowstyle='->', color='#37474F', lw=1.5))

ax.set_title('Celula LSTM: compuertas de memoria selectiva', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('images/B05B_fig03.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05B_fig03.png')

Guardado: images/B05B_fig03.png


## 4. Ejemplos de uso pedagógico

**Ejemplo 1: Clasificación de tickets por urgencia**

Input: secuencia de 5 tickets anteriores del cliente [t-4, t-3, t-2, t-1, t]
Output: probabilidad de que el ticket actual sea urgente
La LSTM aprende que 3 tickets en 24 horas + palabras de escalada = urgente.

**Ejemplo 2: Predicción de siguiente evento**

Input: log de actividad del usuario (login, search, download, logout...)
Output: accion mas probable en el siguiente paso
Util para personalizacion y deteccion de anomalias.

**Ejemplo 3: Series temporales (ver B05A)**

Input: ventana de N meses de churn
Output: churn del mes siguiente

In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

col_labels = [
    'Caso de uso',
    'Long. secuencia',
    'Alternativa simple',
    'Cuando elegir LSTM'
]

rows = [
    ['Clasificacion texto\ncorto (<50 tokens)',
     'corta',
     'BoW + logistic\nregression',
     'No necesario'],
    ['Clasificacion texto\nlargo (>200 tokens)',
     'larga',
     'Transformer',
     'Solo si recursos\nlimitados'],
    ['Series temporales\n(< 50 pasos)',
     'media',
     'Gradient boosting\n+ features temporales',
     'Si hay multiples series\ncorrelacionadas'],
    ['Log de eventos\n(patrones temporales)',
     'variable',
     'Markov chains',
     'Si hay dependencias\nlargas'],
    ['Datos tabulares\nsin orden',
     'N/A',
     'XGBoost,\nrandom forest',
     'Nunca: el orden\nno importa'],
]

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.axis('off')

col_colors = ['#BBDEFB', '#C8E6C9', '#FFF9C4', '#FFCCBC']
row_colors = [['#E3F2FD', '#EDF7ED', '#FFFDE7', '#FBE9E7']] * len(rows)

tbl = ax.table(
    cellText=rows,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    cellColours=row_colors,
    colColours=col_colors
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1.0, 2.2)

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold', fontsize=9)
    cell.set_edgecolor('#BDBDBD')

ax.set_title('Cuando usar RNN/LSTM vs alternativas mas simples',
             fontsize=11, pad=12, y=0.98)
plt.tight_layout()
plt.savefig('images/B05B_fig04.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05B_fig04.png')

Guardado: images/B05B_fig04.png


## 5. Cuando NO usar RNN/LSTM

- Datos tabulares sin secuencia temporal: usar XGBoost/Random Forest
- Texto corto con vocabulario limitado: Bag-of-Words + regresion logística puede bastar
- Secuencias muy largas (>500 pasos): los Transformers son superiores
- Pocos datos: una RNN LSTM necesita miles de secuencias para aprender;
  con pocos datos overfittea

In [6]:
import numpy as np

# LSTM simplificada: compuertas como sigmoide aplicada a combinacion lineal
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# Un paso de LSTM con dimensiones reducidas
np.random.seed(42)
h_size, x_size = 4, 3  # estado oculto y entrada pequenos para ilustrar
x_t    = np.random.randn(x_size)   # input en t
h_prev = np.zeros(h_size)          # estado oculto anterior
c_prev = np.zeros(h_size)          # cell state anterior

W_f = np.random.randn(h_size, h_size + x_size) * 0.1
W_i = np.random.randn(h_size, h_size + x_size) * 0.1
W_c = np.random.randn(h_size, h_size + x_size) * 0.1
W_o = np.random.randn(h_size, h_size + x_size) * 0.1

concat = np.concatenate([h_prev, x_t])
f_t    = sigmoid(W_f @ concat)       # forget gate
i_t    = sigmoid(W_i @ concat)       # input gate
c_hat  = np.tanh(W_c @ concat)       # candidate cell
o_t    = sigmoid(W_o @ concat)       # output gate

c_t = f_t * c_prev + i_t * c_hat    # new cell state
h_t = o_t * np.tanh(c_t)            # new hidden state

print(f'Forget gate (que olvidar): {f_t.round(3)}')
print(f'Input gate  (que anadir):  {i_t.round(3)}')
print(f'Output gate (que mostrar): {o_t.round(3)}')
print(f'Cell state nuevo:          {c_t.round(3)}')
print(f'Hidden state nuevo:        {h_t.round(3)}')

Forget gate (que olvidar): [0.52  0.464 0.474 0.484]
Input gate  (que anadir):  [0.453 0.476 0.471 0.496]
Output gate (que mostrar): [0.509 0.478 0.501 0.489]
Cell state nuevo:          [ 0.008 -0.003 -0.068  0.013]
Hidden state nuevo:        [ 0.004 -0.001 -0.034  0.006]


## Ejercicio técnico

Modifica el codigo anterior para simular 5 pasos de una secuencia.
En cada paso t, alimenta x_t con valores de [0.5, 0.2, -0.3, 0.8, 0.1]
(un valor escalar por paso, replicado como vector de dimension x_size).
Imprime el cell state c_t y el hidden state h_t en cada paso.

Que compuerta parece dominante cuando la entrada es positiva?
Como cambia c_t paso a paso a medida que se acumula información?

In [7]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

np.random.seed(42)
h_size, x_size = 4, 3
h_prev = np.zeros(h_size)
c_prev = np.zeros(h_size)

W_f = np.random.randn(h_size, h_size + x_size) * 0.1
W_i = np.random.randn(h_size, h_size + x_size) * 0.1
W_c = np.random.randn(h_size, h_size + x_size) * 0.1
W_o = np.random.randn(h_size, h_size + x_size) * 0.1

secuencia = [0.5, 0.2, -0.3, 0.8, 0.1]

print(f'  {"Paso":>5}  {"f_t(mean)":>10}  {"i_t(mean)":>10}  {"o_t(mean)":>10}  {"c_t(mean)":>10}')
print('  ' + '-' * 58)

for t, val in enumerate(secuencia):
    x_t    = np.full(x_size, val)
    concat = np.concatenate([h_prev, x_t])
    f_t    = sigmoid(W_f @ concat)
    i_t    = sigmoid(W_i @ concat)
    c_hat  = np.tanh(W_c @ concat)
    o_t    = sigmoid(W_o @ concat)
    c_t    = f_t * c_prev + i_t * c_hat
    h_t    = o_t * np.tanh(c_t)
    print(f'  {t:>5}  {f_t.mean():>10.4f}  {i_t.mean():>10.4f}  {o_t.mean():>10.4f}  {c_t.mean():>10.4f}')
    h_prev, c_prev = h_t, c_t

print()
print('Observacion: el cell state c_t acumula informacion de todos los pasos.')
print('La forget gate controla cuanto del pasado sobrevive a cada paso.')
print('Con entradas positivas, la input gate tiende a ser mas activa.')

   Paso   f_t(mean)   i_t(mean)   o_t(mean)   c_t(mean)
  ----------------------------------------------------------
      0      0.4921      0.5137      0.4870     -0.0085
      1      0.4971      0.5065      0.4948     -0.0066
      2      0.5050      0.4927      0.5078      0.0036
      3      0.4873      0.5216      0.4793     -0.0114
      4      0.4988      0.5042      0.4974     -0.0055

Observacion: el cell state c_t acumula informacion de todos los pasos.
La forget gate controla cuanto del pasado sobrevive a cada paso.
Con entradas positivas, la input gate tiende a ser mas activa.


## Ejercicio de decision

El equipo de datos quiere predecir si un cliente va a cancelar en los proximos 30 dias,
usando el historial de las ultimas 10 interacciones (tickets, logins, pagos).
Proponen una LSTM con 128 unidades.

Es razonable esta propuesta? Que alternativa mas simple probarias primero?
Que necesitas saber sobre el dataset antes de decidir?

**Criterios de evaluacion:** mencionar número de clientes con historial completo,
tasa de churn (si es 5%, hay muy pocos positivos para entrenar),
si el orden de las 10 interacciones aporta información mas alla del simple conteo,
y si un modelo de conteo de eventos + gradient boosting da un baseline razonable.

## Recapitulacion

| Arquitectura | Mecanismo | Limitacion | Cuando |
|---|---|---|---|
| RNN simple | h_t = tanh(W.[h,x]) | Vanishing gradient >20 pasos | Secuencias cortas, texto muy simple |
| LSTM | 3 compuertas de memoria | Costosa computacionalmente | Secuencias largas, series temporales |
| GRU | 2 compuertas (simplificacion) | Menos expresiva que LSTM | Alternativa eficiente a LSTM |
| Transformer | Auto-atencion global | Necesita mucho dato | Estado del arte en texto largo |

## Assets generados

- `images/B05B_fig01.png` -- Celula RNN desenrollada
- `images/B05B_fig02.png` -- Vanishing gradient: |w|^t para distintos valores de w
- `images/B05B_fig03.png` -- Celula LSTM con 3 compuertas
- `images/B05B_fig04.png` -- Tabla: cuando usar RNN/LSTM vs alternativas